# Gambling industry update — FLUT, DKNG and CZR

Read the short update first, then inspect the numerical evidence and sources below. The current snapshot and comparison window are selected once in `config/current_snapshot.json`; no file is chosen by newest timestamp. Run all cells offline. **Amounts are USD; shares and hold are percent; changes are percentage points.**

This answers whether covered-state wagering demand is higher/lower and whether the named brands gain/lose share. It does not produce an overall score, national growth rate, issuer earnings estimate or valuation change. A July-only quarter-to-date window is one of three months; current captures cannot reconstruct what was known historically.


In [ ]:
as_of = None  # None uses current UTC; a historical cutoff rejects later captures.
quarter = None  # None uses the explicit current_snapshot.json window; e.g. "2026Q3"
through_month = None  # e.g. "2026-08" shows missing months rather than shrinking the window.
ny_weeks = 8
export_note = False
note_destination = None  # Optional absolute NEW dated .md path, existing directory.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import sys
import pandas as pd
from IPython.display import display, Markdown

ROOT = Path.cwd().resolve()
if (ROOT / "gaming" / "src").is_dir():
    ROOT = ROOT / "gaming"
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from variant_gaming.refresh import load_validated_snapshot, frozen_database_sha256
selection = json.loads((ROOT / "config/current_snapshot.json").read_text())
DB = ROOT / selection["database_file"]
cutoff = as_of or datetime.now(timezone.utc).isoformat()
snapshot = load_validated_snapshot(root=ROOT, database=DB, as_of=cutoff,
                                   expected_sha256=selection["database_sha256"])
observations, coverage = snapshot["observations"], snapshot["coverage"]
manifest = snapshot["manifest"]
before = snapshot["binding"]["database_sha256"]
pd.set_option("display.max_colwidth", None)
print("Capture:", manifest["finished_at"], "| Information cutoff:", cutoff)
print("Explicit snapshot:", DB)

from variant_gaming.industry import (
    build_industry_tables, business_update, operator_scope, coverage_table,
    research_note, export_research_note,
)
from variant_gaming.flut_scorecard import scorecard_scope
from variant_gaming.legal import load_legal_events
quarter = quarter or selection["quarter"]
through_month = through_month or selection["through_month"]
if pd.Period(through_month, freq="M").end_time.date() > pd.Timestamp(cutoff).date():
    raise ValueError("Requested comparison month has not ended by the information cutoff")
tables = build_industry_tables(observations, quarter=quarter, through_month=through_month, ny_weeks=ny_weeks)
legal_events = load_legal_events(ROOT / "config/legal_events.json", root=ROOT, as_of=cutoff)
update = business_update(tables, legal_events)
print("Window:", quarter, "through", through_month, "versus the same months one year earlier")
display(update)


## Industry demand and competitive position

Official statewide **handle measures wagering volume**, while brand handle share describes position within that state. They are different questions. Every expected month must qualify in both years; a missing August blocks an August-to-date comparison even if July is present. Growth is undefined for a zero or negative baseline.

MA/MI are the measured monthly panel. The full capture contains other state reports with different definitions and limitations; its size does not establish nationwide representativeness.


In [ ]:
display(tables["demand"][["state_code", "native_metric", "expected_months", "market_amount", "prior_market_amount", "market_growth_pct", "direction", "status"]].round(3))
display(tables["competition"][["company", "state_code", "company_amount", "prior_company_amount", "company_growth_pct", "company_share_pct", "prior_company_share_pct", "share_change_pp", "share_direction", "status"]].round(3))


## Sportsbook gross hold and separate casino evidence

Gross hold is native gross revenue divided by handle for exactly the same window. Higher hold is not by itself stronger demand, proven retention, sustainable margin or a company EBITDA increase. MA Accrual Win, MI Gross Receipts, MI Adjusted Gross and MA Taxable Gaming Revenue remain distinct. Casino has no sportsbook handle/hold measure.


In [ ]:
display(tables["hold"].round(3))
display(tables["market_hold"].round(3))
display(tables["market"].query("vertical == 'online_casino'")[["state_code", "native_metric", "market_amount", "prior_market_amount", "market_growth_pct", "direction", "status"]].round(3))
display(tables["casino"][["company", "state_code", "native_metric", "company_amount", "prior_company_amount", "company_growth_pct", "company_share_pct", "share_change_pp", "status"]].round(3))


## New York — timely weekly evidence

Recent complete Monday–Sunday weeks remain weekly. NY reports cash-basis GGR; do not prorate weeks into calendar months or blend this with the monthly panel. Wagering changes across adjacent weeks can reflect sports calendars. Missing weeks, conflicting revisions, invalid handles and unreconciled totals remain visible.


In [ ]:
weekly = tables["weekly"]
if weekly.empty:
    print("New York weekly evidence is unavailable.")
else:
    display(weekly.drop(columns="source_refs").round(3))


## Law and regulation — observed status, then interpretation

This small retained event register is a starting watchlist. **The source's status date is distinct from capture time.** Older enacted tax rules are baseline context; interim rulings are not final nationwide outcomes. Each event records company scope, effective date, observed statutory/accounting effect, inference, limits and the next check. Recheck flags identify unconfirmed current legal status.


In [ ]:
display(legal_events[["event_id", "jurisdiction", "status", "status_as_of", "effective_date", "effective_date_context", "tickers", "applicable_scope", "observed_fact", "accounting_effect", "business_interpretation", "limitations", "recheck_status", "next_check"]])
display(legal_events[["event_id", "source_locator", "published_on", "publication_clock", "captured_at", "source_url", "source_file", "source_sha256"]])


## Drilldown — coverage, exact identities, monthly inputs and sources

FLUT is limited to native FanDuel evidence in these US states. DKNG is the native DraftKings brand/license and excludes Golden Nugget. CZR includes MA/NY sportsbook only; Michigan needs a reviewed multi-license aggregation, and digital evidence cannot represent the land-based group. PA/NJ licensees are not automatically brands. These are coverage limits, not zero activity.

The source coverage table below includes states not selected in the most recent refresh. Their old rows are retained, not recertified by that refresh. Report-period age is an attention flag, not a regulatory deadline.


In [ ]:
display(operator_scope())
display(tables["quarterly"][["company", "state_code", "native_metric", "expected_months", "observed_window_months", "expected_window_months", "matched_window_months", "quarter_complete", "missing_or_excluded_months", "status"]])
q = pd.Period(quarter, freq="Q-DEC")
months = pd.period_range(q.start_time, pd.Period(through_month, freq="M").start_time, freq="M")
input_months = {str(m) for m in months} | {str(m-12) for m in months}
window_inputs = tables["monthly"][tables["monthly"].period_start.str[:7].isin(input_months)]
display(window_inputs.drop(columns="source_refs"))
market_inputs = tables["market_monthly"][tables["market_monthly"].period_start.str[:7].isin(input_months)]
refs = sorted({ref for group in [window_inputs, market_inputs, weekly] for values in group.source_refs for ref in values})
source_table = pd.DataFrame(refs, columns=["source_url", "source_file", "source_sha256"])
display(source_table)
display(scorecard_scope(observations))
display(coverage_table(observations, coverage, capture_at=manifest["finished_at"]))
print("Explicitly selected sources in this capture:")
display(pd.DataFrame(manifest["sources"]))
display(pd.read_csv(DB.parent / "collection_summary.csv"))


## Optional dated research note

Saving descriptive research does not require a forecast or valuation approval. Export is off by default and requires a new dated path. The note includes the measured window, capture and source references. No forecast, trade or model action follows from it.


In [ ]:
note = research_note(update, quarter=quarter, through_month=through_month,
    capture_at=manifest["finished_at"], database_sha256=before, sources=source_table, legal_events=legal_events)
assert frozen_database_sha256(DB) == before, "Database changed during review"
if export_note:
    if note_destination is None:
        raise ValueError("Set an absolute new dated note_destination before enabling export")
    print("Saved:", export_research_note(note, note_destination))
else:
    print("Research note prepared in memory. Export remains disabled.")
